In [1]:
import os
import re

import numpy as np
import torch
import pandas as pd
import random
from val_test import val_test
from print_results import *
import pickle
from tqdm.notebook import tqdm
from functionsgpu_old import *
from plotting_betas import *
from plotting_beta import *
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from video_saving import *

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)
dtype = torch.float32

if device.type == "cuda":
    idx = device.index if device.index is not None else torch.cuda.current_device()
    print(torch.cuda.get_device_name(idx))

SEED = 42
def deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

deterministic(SEED)
# Enable (as much as possible) deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tslen = 200

functionsgpu_old.py device: cuda:1
cuda:1
NVIDIA RTX A5000


## Loading Data

In [2]:
#loading
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all

betas_aligned_all, mu_all, tangent_vec_all = loading('aligned_data',tslen)

# Load y_lesion and participant IDs
# Participant ID for each row of dataset (same order as files from csv_r)
participant_ids = np.loadtxt('pids.txt')
demo_df = pd.read_csv('demo_data.csv')
id_to_lesion = dict(zip(demo_df['s'].astype(int), demo_df['LesionLeft']))
y_lesion = np.array([id_to_lesion[int(pid)] for pid in participant_ids])

print("y_lesion shape:", y_lesion.shape)
print("First 10 participant_ids:", participant_ids[:10])

y_lesion shape: (155,)
First 10 participant_ids: [ 1.  2.  3.  4.  5.  6.  7.  8. 10. 11.]


## Tangent PCA

In [5]:
K = 32
M = 3
T = tslen
R = 38
nsamples = 155

tangent_flat = tangent_vec_all.reshape((K*M*T, nsamples))
U, sigma, V_t = np.linalg.svd(tangent_flat, full_matrices=False)
pc_cords = np.diag(sigma[0:R])@V_t[0:R,:]
X = pc_cords.T
print(X.shape)

(155, 38)


In [6]:
def fpca_project(tangent_flat_train, tangent_flat_test, R=3):
    """Project train and test onto first R left singular vectors of train."""
    U, sigma, Vt = np.linalg.svd(tangent_flat_train, full_matrices=False)
    # Train: coordinates (n_train, R)
    X_train = (Vt[:R].T * sigma[:R])
    # Test: project onto U[:, :R]
    X_test = tangent_flat_test.T @ U[:, :R]
    return X_train, X_test

## PCA Classification

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Same CV: 5 validation + 5 test (disjoint) per fold, two rounds so every subject is validated and tested once.
n_cls = len(y_lesion)
n_folds = 30

models_clf = {'KNN3': KNeighborsClassifier(n_neighbors=3),
              'KNN5': KNeighborsClassifier(n_neighbors=5),
              'KNN7': KNeighborsClassifier(n_neighbors=7),
              'KNN9': KNeighborsClassifier(n_neighbors=9),
              'KNN11': KNeighborsClassifier(n_neighbors=11)}

all_val_clf = {name: {'targets': [], 'preds': []} for name in models_clf.keys()}
all_test_clf = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models_clf.keys()}
participant_ids = np.asarray(participant_ids)

for name, model in tqdm(models_clf.items(), desc='Models'):
    for k in tqdm(range(n_folds), total=n_folds, desc=name, leave=False):
        validation_pids_list, test_pids_list = val_test(participant_ids, k)
        validation_pids = set(validation_pids_list)
        test_pids = set(test_pids_list)
        train_pids = set(participant_ids) - validation_pids - test_pids

        train_idx = np.array([j for j in range(n_cls) if participant_ids[j] in train_pids])
        validation_idx = np.array([j for j in range(n_cls) if participant_ids[j] in validation_pids])
        test_idx = np.array([j for j in range(n_cls) if participant_ids[j] in test_pids])
        if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
            continue

        tangent_flat_train = tangent_flat[:, train_idx]
        tangent_flat_validation = tangent_flat[:, validation_idx]
        tangent_flat_test = tangent_flat[:, test_idx]
        X_train, X_validation = fpca_project(tangent_flat_train, tangent_flat_validation, R=R)
        X_train, X_test = fpca_project(tangent_flat_train, tangent_flat_test, R=R)

        y_train = y_lesion[train_idx]
        m = type(model)(**model.get_params())
        m.fit(X_train, y_train)
        validation_preds = m.predict(X_validation)
        test_preds = m.predict(X_test)
        all_val_clf[name]['targets'].extend(y_lesion[validation_idx].tolist())
        all_val_clf[name]['preds'].extend(validation_preds.tolist())
        all_test_clf[name]['targets'].extend(y_lesion[test_idx].tolist())
        all_test_clf[name]['preds'].extend(test_preds.tolist())
        all_test_clf[name]['subjects'].extend(participant_ids[test_idx].tolist())

        acc_val = accuracy_score(y_lesion[validation_idx], validation_preds)
        f1_val = f1_score(y_lesion[validation_idx], validation_preds, average='weighted')
        # print(f"Fold {k + 1:02d} | {name} | Validation: Accuracy={acc_val:.3f}, F1={f1_val:.3f}")

# print("\n=== LesionLeft Classification — Validation Performance (across all folds) ===")
results_val_clf = {}
for name in models_clf.keys():
    t = np.array(all_val_clf[name]['targets'])
    p = np.array(all_val_clf[name]['preds'])
    if t.size == 0:
        continue
    results_val_clf[name] = {
        'Accuracy': accuracy_score(t, p),
        'F1 (weighted)': f1_score(t, p, average='weighted'),
        'F1 (macro)': f1_score(t, p, average='macro'),
        'Precision (weighted)': precision_score(t, p, average='weighted', zero_division=0),
        'Precision (macro)': precision_score(t, p, average='macro'),
        'Recall (weighted)': recall_score(t, p, average='weighted', zero_division=0),
        'Recall (macro)': recall_score(t, p, average='macro'),
    }
# print(pd.DataFrame(results_val_clf).T)

print("\n=== LesionLeft Classification — Test Performance (across all folds) ===")
results_clf = {}
predictions_clf = {}
for name in models_clf.keys():
    all_targets = np.array(all_test_clf[name]['targets'])
    all_preds = np.array(all_test_clf[name]['preds'])
    if all_targets.size == 0:
        continue
    predictions_clf[name] = {'y_true': all_targets, 'y_pred': all_preds}
    results_clf[name] = {
        'Accuracy': accuracy_score(all_targets, all_preds),
        'F1 (weighted)': f1_score(all_targets, all_preds, average='weighted'),
        'F1 (macro)': f1_score(all_targets, all_preds, average='macro'),
        'Precision (weighted)': precision_score(all_targets, all_preds, average='weighted', zero_division=0),
        'Precision (macro)': precision_score(all_targets, all_preds, average='macro'),
        'Recall (weighted)': recall_score(all_targets, all_preds, average='weighted', zero_division=0),
        'Recall (macro)': recall_score(all_targets, all_preds, average='macro'),
    }
results_clf_df = pd.DataFrame(results_clf).T
results_clf_df

Models:   0%|          | 0/5 [00:00<?, ?it/s]

KNN3:   0%|          | 0/30 [00:00<?, ?it/s]

KNN5:   0%|          | 0/30 [00:00<?, ?it/s]

KNN7:   0%|          | 0/30 [00:00<?, ?it/s]

KNN9:   0%|          | 0/30 [00:00<?, ?it/s]

KNN11:   0%|          | 0/30 [00:00<?, ?it/s]


=== LesionLeft Classification — Test Performance (across all folds) ===


,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
KNN3,0.903226,0.897525,0.800233,0.898051,0.827017,0.903226,0.782540
KNN5,0.896774,0.889882,0.786158,0.892106,0.826981,0.896774,0.758730
KNN7,0.883871,0.875747,0.748646,0.876360,0.786189,0.883871,0.723810
KNN9,0.883871,0.875009,0.741581,0.873123,0.784343,0.883871,0.711111
KNN11,0.877419,0.865107,0.705780,0.862240,0.763580,0.877419,0.674603


In [20]:
from ci_class import *

ci_results = {}
names = models_clf.keys()

for name in tqdm(names):
    ci_results[name] = subject_bootstrap_ci_class(
        all_test_clf[name]['targets'],
        all_test_clf[name]['preds'],
        all_test_clf[name]['subjects']
    )
df = pd.DataFrame(ci_results).T
df['F1 (macro)'].apply(pd.Series)

  0%|          | 0/5 [00:00<?, ?it/s]

,mean,ci
KNN3,0.800233,"[0.723, 0.868]"
KNN5,0.786158,"[0.706, 0.86]"
KNN7,0.748646,"[0.667, 0.826]"
KNN9,0.741581,"[0.654, 0.824]"
KNN11,0.705780,"[0.61, 0.792]"


## KT-RSV Classification

In [21]:
class NonlinearVAE(nn.Module):
    """NonlinearVAE"""
    def __init__(self, D, R, H=128, dropout=0.25):
        super().__init__()
        # Encoder layers
        self.W1 = nn.Linear(D, H, bias=False)        # input -> hidden
        self.W2_mu = nn.Linear(H, R, bias=False)     # hidden -> latent mean
        self.W2_logvar = nn.Linear(H, R)             # hidden -> latent logvar
        self.dropout = nn.Dropout(p=dropout)
        
        # Decoder layers
        self.dec1 = nn.Linear(R, 32, bias=False)
        self.dec2 = nn.Linear(32, D, bias=False)

    def encode(self, x):
        h = torch.tanh(self.W1(x))
        h = self.dropout(h)
        mu = self.W2_mu(h)
        logvar = self.W2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h_recon = torch.tanh(self.dec1(z))
        x_hat = self.dec2(h_recon)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def ktrsv_loss(x_man, x_hat_man, mu, logvar, beta=1e-4):
    dist = squared_geodesic_distance(x_man, x_hat_man, K, M, T)
    recon = dist.mean()
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    avg_kl = kl.mean()
    return recon + beta * avg_kl, recon, avg_kl

In [22]:
class KendallRVAE(nn.Module):
    """rVAE: encode tangent vectors, decode to tangent, but (during
    training) compare reconstructions on the manifold via an exp map.

    - Inputs: tangent vectors (N, D) as in the VAE section.
    - Decoder: produces tangent vectors.
    - Training loss: uses expmap(mu, v_hat) vs. original manifold trajectory.
    """
    def __init__(self, base_vae, mu_shape, expmap):
        super().__init__()
        self.vae = base_vae
        # mean shape, used for exponential map when mapping back to manifold
        self.register_buffer("mu_shape", mu_shape)
        self.expmap = expmap

    def forward(self, x):
        """Forward on tangent vectors.

        x : (N, D) tangent vectors
        Returns
        -------
        x_man_hat : (N, D) manifold trajectory flattened (via expmap)
        mu_z, logvar, z, v_hat : usual VAE outputs (in tangent space)
        """
        v_hat, mu_z, logvar, z = self.vae(x)   # tangent reconstruction

        # Map reconstructed tangent field back to the manifold
        B = v_hat.shape[0]
        v_hat_reshaped = v_hat.view(B, K, M, T)
        mu = self.mu_shape.view(K, M, T)
        x_recon_man = self.expmap(mu, v_hat_reshaped)   # (B, K, M, T)
        x_recon_man = x_recon_man.view(B, -1)

        return x_recon_man, mu_z, logvar, z, v_hat

## Getting Orginal Trajectories on Manifold

In [25]:
betas = np.array(betas_aligned_all)
print(betas.shape)   # (155, 32, 3, 200)

N, K, M, T = betas.shape   # correct order

# Shape mean as before (kept for potential manifold utilities)
mu_shape = torch.from_numpy(
    mu_all.reshape(-1).astype(np.float32)
).to(device)

print("mu_shape:", mu_shape.shape)      # should be (19200,)

(155, 32, 3, 200)
mu_shape: torch.Size([19200])


## Training Function for Each Fold (KT-RSV)

In [26]:
def train_kendall_rvae_fold(X_tan_train, X_man_train, R=10, num_epochs=2000, batch_size=145,
                            lr=1e-3, betakl=2**(-3), seed=42):
    """Train a fresh KendallRVAE on a training subset.

    X_tan_train : (N_train, D) tangent vectors
    X_man_train : (N_train, K*M*T) manifold trajectories (flattened)
    """
    deterministic(seed)
    D = X_tan_train.shape[1]

    dataset = TensorDataset(X_tan_train, X_man_train)
    g = torch.Generator(device=device).manual_seed(seed)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False, generator=g)

    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = KendallRVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)
    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    model_fold.train()
    for epoch in range(num_epochs):
        epoch_loss, epoch_recon, epoch_kl, num_samples = 0.0, 0.0, 0.0, 0

        for (x_tan_batch, x_man_batch) in loader:
            x_tan_batch = x_tan_batch.to(device=device, dtype=dtype)
            x_man_batch = x_man_batch.to(device=device, dtype=dtype)

            opt_fold.zero_grad(set_to_none=True)
            x_hat_man, mu, logvar, z, v_hat = model_fold(x_tan_batch)
            loss_train, recon_train, kl_train = ktrsv_loss(x_man_batch, x_hat_man, mu, logvar, beta=betakl)
            loss_train.backward()
            opt_fold.step()

            bs = x_tan_batch.size(0)
            epoch_loss += loss_train.item() * bs
            epoch_recon += recon_train.item() * bs
            epoch_kl += kl_train.item() * bs
            num_samples += bs

    model_fold.eval()
    return model_fold

## KT-RSV Cross Validation

In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Classification models
models = {'KNN3': KNeighborsClassifier(n_neighbors=3),
              'KNN5': KNeighborsClassifier(n_neighbors=5),
              'KNN7': KNeighborsClassifier(n_neighbors=7),
              'KNN9': KNeighborsClassifier(n_neighbors=9),
              'KNN11': KNeighborsClassifier(n_neighbors=11)}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_lesion)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 30
R = 38
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    deterministic(fold_seed)

    # Slice tangent and manifold data for this fold
    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]

    X_tan_val = X_tan[validation_idx]
    X_man_val = X_man[validation_idx]

    bsize = X_tan_train.shape[0]

    # Train fold-specific KendallRVAE on train participants only
    model_fold = train_kendall_rvae_fold(X_tan_train, X_man_train, R=R, batch_size=bsize,
                                         num_epochs=25, lr=1e-5, seed=fold_seed)

    # Extract latent means (mu) for train, validation, and test using the fold-specific encoder
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_validation_fold, _ = model_fold.vae.encode(X_tan[validation_idx])
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_validation_fold = mu_validation_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    y_train_fold = y_lesion[train_idx]
    y_validation_fold = y_lesion[validation_idx]
    y_test_fold = y_lesion[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        train_preds = m.predict(Z_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold train metrics
        f1_train = f1_score(y_train_fold, train_preds, average='macro')

        # Per-fold validation metrics
        f1_val = f1_score(y_validation_fold, validation_preds, average='macro')

        # Per-fold test metrics
        f1_test = f1_score(y_test_fold, test_preds, average='macro')
        
        print(f"Fold {k + 1:02d} | Train: F1={f1_train:.2f}, Validation: F1={f1_val:.2f},",
              f"Test: F1={f1_test:.2f}")

results_nested_df = print_results_clf(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 01 | Train: F1=0.85, Validation: F1=0.44, Test: F1=0.19
Fold 01 | Train: F1=0.89, Validation: F1=0.30, Test: F1=0.44
Fold 01 | Train: F1=0.81, Validation: F1=0.44, Test: F1=0.44
Fold 01 | Train: F1=0.81, Validation: F1=0.62, Test: F1=0.44
Fold 01 | Train: F1=0.77, Validation: F1=0.44, Test: F1=0.44
Fold 02 | Train: F1=0.89, Validation: F1=0.60, Test: F1=0.43
Fold 02 | Train: F1=0.85, Validation: F1=0.60, Test: F1=0.60
Fold 02 | Train: F1=0.80, Validation: F1=0.19, Test: F1=0.60
Fold 02 | Train: F1=0.86, Validation: F1=0.19, Test: F1=0.60
Fold 02 | Train: F1=0.83, Validation: F1=0.19, Test: F1=0.30
Fold 03 | Train: F1=0.84, Validation: F1=0.43, Test: F1=0.44
Fold 03 | Train: F1=0.83, Validation: F1=0.56, Test: F1=0.44
Fold 03 | Train: F1=0.86, Validation: F1=0.56, Test: F1=0.44
Fold 03 | Train: F1=0.77, Validation: F1=0.43, Test: F1=0.44
Fold 03 | Train: F1=0.73, Validation: F1=0.43, Test: F1=0.44
Fold 04 | Train: F1=0.85, Validation: F1=0.43, Test: F1=0.44
Fold 04 | Train: F1=0.79

,Accuracy,F1 (macro),Precision (macro),Recall (macro)
KNN3,0.883871,0.742985,0.774128,0.723810
KNN5,0.916129,0.825390,0.856002,0.804762
KNN7,0.903226,0.789618,0.833349,0.757143
KNN9,0.909677,0.805866,0.859642,0.768254
KNN11,0.896774,0.766184,0.813488,0.733333


In [28]:
from ci_class import *

ci_results = {}
names = models_clf.keys()

for name in tqdm(names):
    ci_results[name] = subject_bootstrap_ci_class(
        all_results_nested[name]['targets'],
        all_results_nested[name]['preds'],
        all_results_nested[name]['subjects']
    )
df = pd.DataFrame(ci_results).T
df['F1 (macro)'].apply(pd.Series)

  0%|          | 0/5 [00:00<?, ?it/s]

,mean,ci
KNN3,0.742985,"[0.657, 0.824]"
KNN5,0.825390,"[0.754, 0.897]"
KNN7,0.789618,"[0.708, 0.864]"
KNN9,0.805866,"[0.722, 0.881]"
KNN11,0.766184,"[0.68, 0.845]"
